In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.base import TransformerMixin, BaseEstimator
from tqdm import tqdm
from gensim.test.utils import common_texts
from gensim.models import Word2Vec

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/test-data/sample_submission.csv
/kaggle/input/test-data/test.csv
/kaggle/input/amazon-fine-food-reviews/hashes.txt
/kaggle/input/amazon-fine-food-reviews/Reviews.csv
/kaggle/input/amazon-fine-food-reviews/database.sqlite


In [2]:
review = pd.read_csv("/kaggle/input/amazon-fine-food-reviews/Reviews.csv")
test = pd.read_csv("/kaggle/input/test-data/test.csv")
review = review.head(10000)
review = review[['Text', 'Score']]
review["label"] = (review["Score"] >= 4).astype(int)
review.head()

,Text,Score,label
0,I have bought several of the Vitality canned d...,5,1
1,Product arrived labeled as Jumbo Salted Peanut...,1,0
2,This is a confection that has been around a fe...,4,1
3,If you are looking for the secret ingredient i...,2,0
4,Great taffy at a great price. There was a wid...,5,1


In [3]:
_token_pattern = re.compile(r"[A-Za-z'-]+")

def simple_tokenize(text):
    text = str(text).lower()
    tokens = _token_pattern.findall(text)
    return tokens
tqdm.pandas()
review['tokens'] = review['Text'].progress_apply(simple_tokenize)
review.head()

100%|██████████| 10000/10000 [00:00<00:00, 45181.08it/s]


,Text,Score,label,tokens
0,I have bought several of the Vitality canned d...,5,1,"[i, have, bought, several, of, the, vitality, ..."
1,Product arrived labeled as Jumbo Salted Peanut...,1,0,"[product, arrived, labeled, as, jumbo, salted,..."
2,This is a confection that has been around a fe...,4,1,"[this, is, a, confection, that, has, been, aro..."
3,If you are looking for the secret ingredient i...,2,0,"[if, you, are, looking, for, the, secret, ingr..."
4,Great taffy at a great price. There was a wid...,5,1,"[great, taffy, at, a, great, price, there, was..."


In [4]:
stop_words = set(ENGLISH_STOP_WORDS)
def tokens_to_str(tokens):
    filtered = [t for t in tokens if t not in stop_words]
    return " ".join(filtered)
review['text_NoStopWord'] = review['tokens'].progress_apply(tokens_to_str)
review.head()

100%|██████████| 10000/10000 [00:00<00:00, 118058.95it/s]


,Text,Score,label,tokens,text_NoStopWord
0,I have bought several of the Vitality canned d...,5,1,"[i, have, bought, several, of, the, vitality, ...",bought vitality canned dog food products good ...
1,Product arrived labeled as Jumbo Salted Peanut...,1,0,"[product, arrived, labeled, as, jumbo, salted,...",product arrived labeled jumbo salted peanuts p...
2,This is a confection that has been around a fe...,4,1,"[this, is, a, confection, that, has, been, aro...",confection centuries light pillowy citrus gela...
3,If you are looking for the secret ingredient i...,2,0,"[if, you, are, looking, for, the, secret, ingr...",looking secret ingredient robitussin believe g...
4,Great taffy at a great price. There was a wid...,5,1,"[great, taffy, at, a, great, price, there, was...",great taffy great price wide assortment yummy ...


In [5]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(review['text_NoStopWord'])
y = review['label'].values
print("TF-IDF shape:", X_tfidf.shape)
tfidf.get_feature_names_out()

TF-IDF shape: (10000, 20000)


array(['aa', 'ability', 'able', ..., 'zucchini', 'zuke', 'zukes'],
      dtype=object)

In [6]:
# 以 tokens (list of list) 作為訓練資料
sentences = review['text_NoStopWord'].tolist()

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,    
    window=5,
    min_count=1,        
    workers=4,
    seed=42
)

# 將每篇評論轉為「平均詞向量」
def doc_vector(tokens):
    vecs = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]
    if len(vecs) == 0:
        return np.zeros(w2v_model.vector_size)
    return np.mean(vecs, axis=0)

X_w2v = np.vstack(review['text_NoStopWord'].progress_apply(doc_vector).values)
print("Word2Vec shape:", X_w2v.shape)

100%|██████████| 10000/10000 [00:04<00:00, 2465.75it/s]

Word2Vec shape: (10000, 100)


In [7]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

# TF-IDF + RF
print("\nRunning 4-fold CV for RandomForest with TF-IDF features ...")
scores_tfidf = cross_val_score(clf, X_tfidf, y, cv=cv, scoring='accuracy', n_jobs=1)
print("TF-IDF Accuracy (4-fold):", scores_tfidf)
print("TF-IDF Accuracy mean = {:.4f}, std = {:.4f}".format(scores_tfidf.mean(), scores_tfidf.std()))


Running 4-fold CV for RandomForest with TF-IDF features ...
TF-IDF Accuracy (4-fold): [0.8276 0.8212 0.824  0.8304]
TF-IDF Accuracy mean = 0.8258, std = 0.0035


In [8]:
# Word2Vec + RF
print("\nRunning 4-fold CV for RandomForest with Word2Vec-avg features ...")
scores_w2v = cross_val_score(clf, X_w2v, y, cv=cv, scoring='accuracy', n_jobs=1)
print("Word2Vec Accuracy (4-fold):", scores_w2v)
print("Word2Vec Accuracy mean = {:.4f}, std = {:.4f}".format(scores_w2v.mean(), scores_w2v.std()))


Running 4-fold CV for RandomForest with Word2Vec-avg features ...
Word2Vec Accuracy (4-fold): [0.784  0.7848 0.786  0.788 ]
Word2Vec Accuracy mean = 0.7857, std = 0.0015


In [9]:
X_test_tfidf = tfidf.transform(test)
clf.fit(X_tfidf, review["label"])

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [10]:
y_pred = clf.predict(X_test_tfidf)

In [11]:
submission = pd.DataFrame({
    'id': range(1, len(y_pred) + 1),
    'pred_label': y_pred
})